# Regularized Logistic Regression

This notebook trains `reg_logistic_regression` on the processed data prepared in notebook 01.

The original class distribution is kept during training and validation. We add element-wise squared features to capture simple nonlinear effects without the memory cost of all pairwise polynomial interactions. Hyperparameters are selected using validation F1 after threshold tuning.

In [1]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'implementations.py').exists():
    PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from implementations import reg_logistic_regression

In [2]:
DATA_PATH = PROJECT_ROOT / 'dataset/prepared/selected_random_over_dataset.npz'
data = np.load(DATA_PATH, allow_pickle=True)

x_train = data['x_train_raw']
y_train = data['y_train_raw'].astype(int)
x_test = data['x_test']
test_ids = data['test_ids']

# Add element-wise quadratic terms; full pairwise expansion would be too large.
x_train_model = np.c_[x_train, x_train**2]
x_test_model = np.c_[x_test, x_test**2]
tx = np.c_[np.ones((x_train_model.shape[0], 1)), x_train_model]
tx_test = np.c_[np.ones((x_test_model.shape[0], 1)), x_test_model]

print(f'Train shape: {tx.shape}')
print(f'Test shape: {tx_test.shape}')
print(f'Class counts: {np.bincount(y_train)}')

Train shape: (328135, 281)
Test shape: (109379, 281)
Class counts: [299160  28975]


In [3]:
def stratified_split(x, y, validation_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    fit_indices = []
    validation_indices = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        rng.shuffle(indices)
        validation_size = int(len(indices) * validation_fraction)
        validation_indices.extend(indices[:validation_size])
        fit_indices.extend(indices[validation_size:])
    rng.shuffle(fit_indices)
    rng.shuffle(validation_indices)
    return x[fit_indices], y[fit_indices], x[validation_indices], y[validation_indices]


def sigmoid(scores):
    scores = np.clip(scores, -500, 500)
    return 1 / (1 + np.exp(-scores))


def metrics(y_true, y_pred):
    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    false_positive = np.sum((y_true == 0) & (y_pred == 1))
    false_negative = np.sum((y_true == 1) & (y_pred == 0))
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1


def best_threshold(y_true, probabilities):
    thresholds = np.unique(probabilities)
    f1_scores = [metrics(y_true, probabilities >= threshold)[2] for threshold in thresholds]
    return float(thresholds[np.argmax(f1_scores)])

In [4]:
tx_fit, y_fit, tx_validation, y_validation = stratified_split(tx, y_train)

lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
gammas = [0.005, 0.01, 0.02, 0.05, 0.1]
iteration_options = [300, 600, 1000]
best_result = None
results = []

for lambda_ in lambdas:
    for gamma in gammas:
        for max_iters in iteration_options:
            weights, loss = reg_logistic_regression(
                y_fit,
                tx_fit,
                lambda_,
                np.zeros(tx_fit.shape[1]),
                max_iters,
                gamma,
            )
            validation_probabilities = sigmoid(tx_validation @ weights)
            threshold = best_threshold(y_validation, validation_probabilities)
            predictions = validation_probabilities >= threshold
            precision, recall, f1 = metrics(y_validation, predictions)
            result = (f1, lambda_, gamma, max_iters, threshold, loss, precision, recall)
            results.append(result)
            if best_result is None or result[0] > best_result[0]:
                best_result = result

results.sort(reverse=True)
(
    best_f1,
    best_lambda,
    best_gamma,
    best_max_iters,
    threshold,
    validation_loss,
    precision,
    recall,
) = best_result

print(f'Best lambda: {best_lambda}')
print(f'Best gamma: {best_gamma}')
print(f'Best iterations: {best_max_iters}')
print(f'Validation threshold: {threshold:.4f}')
print(f'Validation loss: {validation_loss:.4f}')
print(f'Validation precision: {precision:.4f}')
print(f'Validation recall: {recall:.4f}')
print(f'Validation F1: {best_f1:.4f}')
print('\nTop 5 configurations:')
for result in results[:5]:
    print(
        f'F1={result[0]:.4f}, lambda={result[1]}, '
        f'gamma={result[2]}, iterations={result[3]}, threshold={result[4]:.4f}'
    )

Best lambda: 0.0001
Best gamma: 0.1
Best iterations: 1000
Validation threshold: 0.1947
Validation loss: 0.2247
Validation precision: 0.3475
Validation recall: 0.5412
Validation F1: 0.4232

Top 5 configurations:
F1=0.4232, lambda=0.0001, gamma=0.1, iterations=1000, threshold=0.1947
F1=0.4231, lambda=1e-05, gamma=0.1, iterations=1000, threshold=0.1953
F1=0.4224, lambda=0.001, gamma=0.1, iterations=1000, threshold=0.1923
F1=0.4179, lambda=1e-05, gamma=0.1, iterations=600, threshold=0.1761
F1=0.4177, lambda=0.0001, gamma=0.1, iterations=600, threshold=0.1749


In [5]:
# Retrain on the full training data with the selected hyperparameters.
final_weights, final_loss = reg_logistic_regression(
    y_train,
    tx,
    best_lambda,
    np.zeros(tx.shape[1]),
    best_max_iters,
    best_gamma,
)

test_probabilities = sigmoid(tx_test @ final_weights)
test_predictions = np.where(test_probabilities >= threshold, 1, -1)
print(f'Final training loss: {final_loss:.4f}')
print(f'Predicted positive rate: {np.mean(test_predictions == 1):.4f}')

Final training loss: 0.2247
Predicted positive rate: 0.1372


In [6]:
from helpers import create_csv_submission

submission_path = PROJECT_ROOT / 'submission_regularized_logistic.csv'
create_csv_submission(test_ids, test_predictions, submission_path)
print(f'Submission written to: {submission_path}')

Submission written to: /Users/mohamedzirar/Documents/year4/ML/ML_project1/submission_regularized_logistic.csv
